##  Fitting Observed Data

In [ ]:
import sys
sys.path.append('..')
from pathlib import Path
import os
import pandas as pd
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import matplotlib.pyplot as plt
import tarnished_ww
from tarnished_ww.api import fit_joint_model
import arviz as az

# filtering dates if needed
min_date = pd.Timestamp(2023, 1, 1)
max_date = pd.Timestamp(2025, 2, 1)
# df['surveillance_date'] = pd.to_datetime(df['surveillance_date'])
# df = df[(df['surveillance_date'] >= min_date) & (df['surveillance_date'] < max_date)] 
# df = df.sort_values(["wwtp_id", "surveillance_date"])

diseases = ["covid", "rsv", "flua"]

root_dir = Path(os.path.dirname(Path.cwd()))
filepath = os.path.join(root_dir, "data", "processed")

pop = pd.read_csv(os.path.join(filepath, "population.csv"),index_col=[0]).reset_index()
df = pd.read_csv(os.path.join(filepath,"input_data.csv"),index_col=[0]).reset_index()

df["rank_desc"] = df.groupby("wwtp_id")["surveillance_date"].rank(method="first", ascending=False)
# Split into train and test
df_test = df[df["rank_desc"] <= 21].copy()
df_train = df[df["rank_desc"] > 21].copy()

# Drop helper column
df_train.drop(columns="rank_desc", inplace=True)
df_test.drop(columns="rank_desc", inplace=True)

pivot_df = df_train.pivot(index='surveillance_date', columns='wwtp_id', values=['total_ed_visits'])
pivot_df['total_ed_visits'] = pivot_df['total_ed_visits'].fillna(0)
pivot_df = pivot_df.sort_index()
y_ed = pivot_df['total_ed_visits'].values 
y_ed = np.asarray(y_ed)
wwtps = pivot_df.columns.get_level_values(1).unique().tolist()
total_tests_df = df_train.pivot(index='surveillance_date', columns='wwtp_id', values=['total_tests_all_ages'])
total_tests_df = total_tests_df.sort_index()
total_tests_df = total_tests_df.interpolate(method="linear", axis=0, limit_direction="both")
total_tests_per_capita = np.asarray(total_tests_df['total_tests_all_ages'].values)/pop.population.values

In [ ]:
with pm.Model() as joint_model:
    build_joint_model(diseases, df_train, y_ed, pop.population.values, pop.shape[0], tests_per_capita = total_tests_per_capita)
    trace_ed = pm.sample(2000, return_inferencedata=True, target_accept=0.95,random_seed=123)

In [ ]:
from datetime import date

# get today's date in YYYY-MM-DD format
today_str = date.today().strftime("%Y-%m-%d")

with joint_model:
   trace_ed.extend(pm.sample_posterior_predictive(trace_ed,random_seed=123))
trace_ed.to_netcdf(f"ed_tests_{today_str}.nc")


In [ ]:
az.summary(trace_ed, var_names = [ "arrival_rate_free_flua", "arrival_rate_free_rsv",
                                  "arrival_rate_free_covid", "sigma_wastewater_covid", "sigma_wastewater_flua",
                                  "sigma_wastewater_rsv"])

In [ ]:
az.summary(trace_ed, var_names = ["chol_cov_covid", "chol_cov_flua", "chol_cov_rsv"])

In [ ]:
az.summary(trace_ed, var_names = ["nu_covid", "nu_flua", "nu_rsv","offset_ed_covid", "offset_ed_flua", "offset_ed_rsv"])

In [ ]:
az.summary(trace_ed, var_names = ["sd_latent", "alpha_flua","alpha_rsv","alpha_covid"])#,"alpha_test","sigma_pi", "sigma_T"])

In [ ]:
az.summary(trace_ed, var_names = ["beta_tests_covid", "beta_tests_flua", "beta_tests_rsv"])#,"alpha_test","sigma_pi", "sigma_T"])

In [ ]:
az.summary(trace_ed, var_names = ["latent_covid"],coords={"latent_covid_dim_0": slice(200,220), "latent_covid_dim_1": [1]})

In [ ]:
az.summary(trace_ed, var_names = ["latent_ed"],coords={"latent_ed_dim_1": slice(100, 120), "latent_ed_dim_0": [0]})

In [ ]:
t_r_indices = [(100, 0), (200, 1), (300, 2)]

for t, r in t_r_indices:
    var_name = f"latent_ed[{t},{r}]"
    az.plot_trace(trace_ed, var_names=["latent_ed"], coords={"latent_ed_dim_0": [r], "latent_ed_dim_1": [t]})
    plt.suptitle(f"Trace plot for {var_name}")
    plt.show()

In [ ]:
import arviz as az
az.plot_trace(trace_ed, var_names=["latent_ed"], coords={"latent_ed_dim_1": [100, 220, 340], "latent_ed_dim_0": [0]});
az.plot_rank(trace_ed, var_names=["latent_ed"], coords={"latent_ed_dim_1": [300], "latent_ed_dim_0": [0]});


In [ ]:
az.plot_rank(trace_ed, var_names=["latent_rsv"], coords={"latent_rsv_dim_0": [100], "latent_rsv_dim_1": [0]});


In [ ]:
az.plot_pair(trace_ed, var_names=["latent_ed", "latent_rsv"], kind="scatter", coords={"latent_ed_dim_1": [100], "latent_ed_dim_0": [0]});


In [ ]:
az.summary(trace_ed, var_names=["latent_ed"], coords={"latent_ed_dim_1": [100], "latent_ed_dim_0": [0]})

In [ ]:
# Extract posterior samples
samples = trace_ed.posterior["latent_ed"]  # shape: (chain, draw, T, R)
# Compute std across chains and draws
std_latent = samples.stack(sample=("chain", "draw")).std(dim="sample")  # shape: (T, R)

# Plot std over time for one region (e.g., region 0)
plt.plot(std_latent[:, 0])
plt.title("Posterior Std of latent_ed over time (Region 0)")
plt.xlabel("Time")
plt.ylabel("Std Dev")
plt.grid(True)
plt.show()

# Optionally, summarize across time and regions
print("Global latent_ed std mean:", std_latent.mean().item())
print("Global latent_ed std max:", std_latent.max().item())

In [ ]:
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import cholesky, LinAlgError
posterior = trace_ed

L_samples = posterior.posterior["chol_cov_covid_corr"].stack(sample=("chain", "draw")).values
# Reshape to (samples, 4, 4)
L_samples = L_samples.reshape(-1, 4, 4)  # Combine chain and draw

# Compute correlation matrices: corr = L @ L.T
cov_matrices = np.matmul(L_samples, np.transpose(L_samples, (0, 2, 1)))

# Convert each to correlation
corr_matrices = []
for cov in cov_matrices:
    stds = np.sqrt(np.diag(cov))
    corr = cov / stds[:, None] / stds[None, :]
    corr_matrices.append(corr)

corr_matrices = np.array(corr_matrices)

# Compute posterior mean correlation
mean_corr = np.mean(corr_matrices, axis=0)

# Plot heatmap of mean correlation
plt.figure(figsize=(6, 5))
sns.heatmap(mean_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            cbar_kws={"label": "Correlation"})
plt.title("Posterior Mean Correlation Matrix (COVID)")
plt.tight_layout()
plt.show()

In [ ]:
total_tests = np.asarray(total_tests_df['total_tests_all_ages'].values)
plt.plot(total_tests)

In [ ]:
std_corr = np.std(corr_matrices, axis=0)

plt.figure(figsize=(6, 5))
sns.heatmap(std_corr, annot=True, fmt=".2f", cmap="viridis",
            cbar_kws={"label": "Std Dev"})
plt.title("Posterior Std Dev of Correlations (COVID)")
plt.tight_layout()
plt.show()


In [ ]:
az.plot_pair(trace_ed, var_names=["sigma_ed", "chol_cov_flua"], divergences= True);

In [ ]:
az.plot_pair(trace_ed, var_names=["arrival_rate_covid", "arrival_rate_flua", "arrival_rate_rsv"], divergences= True, kind = "kde");

# Other plots

In [ ]:
chol_flat = trace_ed.posterior["chol_cov_covid"].mean(dim=["chain", "draw"]).values  # shape (10,) for 4x4

chol_lower = pm.expand_packed_triangular(4, chol_flat, lower=True).eval()
Sigma = pt.dot(chol_lower, chol_lower.T)
Sigma.eval()

import matplotlib.pyplot as plt
wwtps = ['Annacis','Lulu','Iona','Lionsgate']

plt.figure(figsize=(5, 4))
plt.imshow(Sigma.eval(), cmap='viridis', interpolation='nearest')
plt.colorbar(label='Covariance')
plt.title('Covariance Matrix Heatmap')
plt.xlabel('Dimension')
plt.ylabel('Dimension')
plt.xticks(range(len(wwtps)), wwtps, rotation=45)
plt.yticks(range(len(wwtps)), wwtps)
plt.grid(False)
plt.show()

### ED Visits plots

In [ ]:
diseases = ["covid", "rsv", "flua"]
num_regions = len(wwtps)
pivot_df = df_train.pivot(index='surveillance_date', columns='wwtp', values=['total_ed_visits'])
pivot_df['total_ed_visits'] = pivot_df['total_ed_visits'].fillna(0)
pivot_df = pivot_df.sort_index()
y_ed = pivot_df['total_ed_visits'].values  # shape (T, R)
y_ed = np.asarray(y_ed)

ed_visits_pp =  trace_ed.posterior_predictive[f"ed_visits"]
ed_visits_mean = ed_visits_pp.values.mean(axis=(0,1))  # shape: (T, regions)
ed_visits_hdi = az.hdi(ed_visits_pp.values, hdi_prob=0.95)  # shape: (T(4,2), regions, 2)

pivot_df = df_train.pivot(index='surveillance_date', columns='wwtp', values=['total_ed_visits'])
pivot_df['total_ed_visits'] = pivot_df['total_ed_visits'].fillna(0)
pivot_df = pivot_df.sort_index()
dates = pivot_df.index.values

latent_ed = trace_ed.posterior[f"latent_ed"]
latent_ed_mean = latent_ed.values.mean(axis=(0,1)).T  # shape: (T, regions)
latent_ed_hdi = az.hdi(latent_ed.values, hdi_prob=0.95).transpose(1,0,2)  # shape: (T(4,2), regions, 2)

for r in range(num_regions):
    fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    ax1=ax[0]
    # --- CASES ---
    ax1.plot(dates[1:], y_ed[1:, r], label='Observed ED', color='black')
    ax1.plot(dates[1:], ed_visits_mean[1:, r], label='Predicted Mean', color='blue')
    ax1.fill_between(
        dates[1:],
        ed_visits_hdi[1:, r, 0],
        ed_visits_hdi[1:, r, 1],
        color='blue',
        alpha=0.3,
        label='95% Credible Interval'
    )

    ax2=ax[1]
    # --- CASES ---
    ax2.plot(dates[1:], latent_ed_mean[1:, r], label='Latent', color='red')
    ax2.fill_between(
        dates[1:],
        latent_ed_hdi[1:, r, 0],
        latent_ed_hdi[1:, r, 1],
        color='red',
        alpha=0.3,
        label='95% Credible Interval'
    )
    ax1.set_ylabel("ED Visits", color='black', fontsize=18)
    #ax2.set_ylabel("Predicted Cases", color='blue')
    ax1.set_title(f"Region {wwtps[r]}: ED Visits", fontsize = 24)
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.set_xlabel("Date", fontsize = 22)

## Cases and Wastewater plots

In [ ]:
# Extract posterior predictive samples
from TARnISHED_WW.plots_functions import plot_posterior_data
from TARnISHED_WW.build_functions import getting_df_data_logged
suffix = "flua"
cases_pp = trace_ed.posterior_predictive[f"observed_cases_{suffix}"]
waste_pp =  trace_ed.posterior_predictive[f"observed_wastewater_{suffix}"]
latent = trace_ed.posterior[f"latent_{suffix}"]

# Compute mean and HDI over samples
cases_pp_mean = cases_pp.values.mean(axis=(0,1))  # shape: (T, regions)
cases_pp_hdi = az.hdi(cases_pp.values, hdi_prob=0.95)  # shape: (T(4,2), regions, 2)

waste_pp_mean = waste_pp.values.mean(axis=(0,1))
waste_pp_hdi = az.hdi(waste_pp.values, hdi_prob=0.95)

# Compute mean and HDI over samples
latent_mean = latent.values.mean(axis=(0,1))  # shape: (T, regions)
latent_hdi = az.hdi(latent.values, hdi_prob=0.95) # shape: (T(4,2), regions, 2)

y_cases, log_y_signal_masked, pivot_df = getting_df_data_logged(df_train, suffix)

wwtps = ['Annacis','Lulu','Iona','Lionsgate']

dates = pivot_df.index.values

plot_posterior_data(wwtps,dates, 
                    y_cases, cases_pp_mean, cases_pp_hdi,
                    log_y_signal_masked, waste_pp_mean, waste_pp_hdi,
                    latent_mean, latent_hdi, suffix, forecasting=False, forecast_start = None, latent_plot=True)

In [ ]:
# Extract posterior predictive samples
suffix = "rsv"
cases_pp = trace_ed.posterior_predictive[f"observed_cases_{suffix}"]
waste_pp =  trace_ed.posterior_predictive[f"observed_wastewater_{suffix}"]
latent = trace_ed.posterior[f"latent_{suffix}"]

# Compute mean and HDI over samples
cases_pp_mean = cases_pp.values.mean(axis=(0,1))  # shape: (T, regions)
cases_pp_hdi = az.hdi(cases_pp.values, hdi_prob=0.95)  # shape: (T(4,2), regions, 2)

waste_pp_mean = waste_pp.values.mean(axis=(0,1))
waste_pp_hdi = az.hdi(waste_pp.values, hdi_prob=0.95)

# Compute mean and HDI over samples
latent_mean = latent.values.mean(axis=(0,1)) # shape: (T, regions)
latent_hdi = az.hdi(latent.values, hdi_prob=0.95)  # shape: (T(4,2), regions, 2)

y_cases, log_y_signal_masked, pivot_df = getting_df_data_logged(df_train, suffix)

wwtps = ['Annacis','Lulu','Iona','Lionsgate']

dates = pivot_df.index.values

plot_posterior_data(wwtps,dates, 
                    y_cases, cases_pp_mean, cases_pp_hdi,
                    log_y_signal_masked, waste_pp_mean, waste_pp_hdi,
                    latent_mean, latent_hdi, suffix, forecasting=False, forecast_start = None)

In [ ]:
# Extract posterior predictive samples
suffix = "covid"
cases_pp = trace_ed.posterior_predictive[f"observed_cases_{suffix}"]
waste_pp = trace_ed.posterior_predictive[f"observed_wastewater_{suffix}"]
latent = trace_ed.posterior[f"latent_{suffix}"]

# Compute mean and HDI over samples
cases_pp_mean = cases_pp.values.mean(axis=(0,1))  # shape: (T, regions)
cases_pp_hdi = az.hdi(cases_pp.values, hdi_prob=0.95)  # shape: (T(4,2), regions, 2)

waste_pp_mean = waste_pp.values.mean(axis=(0,1))
waste_pp_hdi = az.hdi(waste_pp.values, hdi_prob=0.95)

# Compute mean and HDI over samples
latent_mean = latent.values.mean(axis=(0,1))  # shape: (T, regions)
latent_hdi = az.hdi(latent.values, hdi_prob=0.95)  # shape: (T(4,2), regions, 2)

wwtps = ['Annacis','Lulu','Iona','Lionsgate']

dates = pivot_df.index.values

y_cases, log_y_signal_masked, pivot_df = getting_df_data_logged(df_train, suffix)

plot_posterior_data(wwtps,dates, 
                    y_cases, cases_pp_mean, cases_pp_hdi,
                    log_y_signal_masked, waste_pp_mean, waste_pp_hdi,
                    latent_mean, latent_hdi, suffix, forecasting=False, forecast_start = None)

# Forecasting

In [ ]:
from TARnISHED_WW.forecast_functions import build_forecast_model, getting_df_data_logged
tests_df = df_test.pivot(index='surveillance_date', columns='wwtp', values=['total_tests_all_ages'])
tests_df = tests_df.sort_index()
tests_df = tests_df.interpolate(method="linear", axis=0, limit_direction="both")
tests_per_capita = np.asarray(tests_df['total_tests_all_ages'].values)/pop.population.values

diseases = ["covid", "rsv", "flua"]
num_regions = len(wwtps)

#trace_ed = az.from_netcdf(f"test1.nc")

pivot_df = df_test.pivot(index='surveillance_date', columns='wwtp', values=['total_ed_visits'])
pivot_df['total_ed_visits'] = pivot_df['total_ed_visits'].fillna(0)
pivot_df = pivot_df.sort_index()

# Extract observed case counts (Poisson likelihood)
y_ed = pivot_df['total_ed_visits'].values  # shape (T, R)
y_ed = np.asarray(y_ed)
with pm.Model() as joint_forecast_model:
    var_names = build_forecast_model(diseases, trace_ed, df_test, y_ed, population, num_regions, tests_per_capita = tests_per_capita, hurdle = False)
    pp_forecast = pm.sample_posterior_predictive(trace_ed, model=joint_forecast_model, predictions=True, var_names=var_names + ["ed_contribution_residual", 
                                                                                                                                "ed_contribution_covid",
                                                                                                                                "ed_contribution_rsv",
                                                                                                                                "ed_contribution_flua"],random_seed=123)
pp_forecast.to_netcdf(f"forecast_ed_tests_{today_str}.nc")

In [ ]:
from TARnISHE_WW.plots_functions import plot_posterior_data 
posterior_predictive = trace_ed
# Extract posterior predictive samples
suffix = "rsv"
cases_pp = posterior_predictive.posterior_predictive[f"observed_cases_{suffix}"]
waste_pp =  posterior_predictive.posterior_predictive[f"observed_wastewater_{suffix}"]
latent = posterior_predictive.posterior[f"latent_{suffix}"]

# Compute mean and HDI over samples
cases_pp_mean = np.median(cases_pp.values, axis=(0,1))  # shape: (T, regions)
cases_pp_hdi = az.hdi(cases_pp.values, hdi_prob=0.95)  # shape: (T(4,2), regions, 2)

waste_pp_mean = np.median(waste_pp.values, axis=(0,1))
waste_pp_hdi = az.hdi(waste_pp.values, hdi_prob=0.95)

# Compute mean and HDI over samples
latent_mean = latent.values.mean(axis=(0,1))  # shape: (T, regions)
latent_hdi = az.hdi(latent.values, hdi_prob=0.95)  # shape: (T(4,2), regions, 2)

# Extract forecasted posterior predictive samples
forecast_cases_pp = pp_forecast.predictions[f"forecast_observed_cases_{suffix}"]  # shape: (chains, draws, H, regions)
forecast_ww_pp = pp_forecast.predictions[f"forecast_observed_wastewater_{suffix}"]
forecast_latent_pp = pp_forecast.predictions[f"latent_forecast_{suffix}"] # shape (chain, draws, horizon, region)

# Compute means and 95% HDIs
forecast_cases_mean = np.median(forecast_cases_pp.values, axis = (0,1)) # shape: (H, R)
forecast_cases_hdi = az.hdi(forecast_cases_pp.values, hdi_prob=0.95)  # shape: (H, R, 2)

forecast_ww_mean = np.median(forecast_ww_pp.values, axis = (0,1))
forecast_ww_hdi = az.hdi(forecast_ww_pp.values, hdi_prob=0.95)

forecast_latent_mean = np.median(forecast_latent_pp.values,axis=(0,1))
forecast_latent_hdi = az.hdi(forecast_latent_pp.values, hdi_prob=0.95)

# Concatenate with posterior predictive values
cases_full_mean = np.concatenate([cases_pp_mean, forecast_cases_mean], axis=0)  # (T+H, R)
cases_full_hdi = np.concatenate([cases_pp_hdi, forecast_cases_hdi], axis=0)

waste_full_mean = np.concatenate([waste_pp_mean, forecast_ww_mean], axis=0)
waste_full_hdi = np.concatenate([waste_pp_hdi, forecast_ww_hdi], axis=0)

latent_full_mean = np.concatenate([latent_mean, forecast_latent_mean], axis=0)
latent_full_hdi = np.concatenate([latent_hdi, forecast_latent_hdi], axis=0)

y_cases, log_y_signal_masked, pivot_df = getting_df_data_logged(df_train, suffix)
y_cases_test, log_y_signal_masked_test, pivot_df_test = getting_df_data_logged(df_test, suffix)

y_cases_full = np.concatenate([y_cases, y_cases_test], axis=0)
log_y_signal_masked_full = np.concatenate([log_y_signal_masked, log_y_signal_masked_test], axis=0)
all_dates = np.concatenate([pivot_df.index.values, pivot_df_test.index.values])

plot_posterior_data(wwtps,all_dates, 
                    y_cases_full, cases_full_mean, cases_full_hdi,
                    log_y_signal_masked_full, waste_full_mean, waste_full_hdi,
                    latent_full_mean, latent_full_hdi, suffix, forecasting=True, forecast_start = pivot_df_test.index[0], 
                    latent_plot=True)

In [ ]:
pivot_df = df_train.pivot(index='surveillance_date', columns='wwtp', values=['total_ed_visits'])
pivot_df['total_ed_visits'] = pivot_df['total_ed_visits'].fillna(0)
pivot_df = pivot_df.sort_index()
pivot_df_test = df_test.pivot(index='surveillance_date', columns='wwtp', values=['total_ed_visits'])
pivot_df_test['total_ed_visits'] = pivot_df_test['total_ed_visits'].fillna(0)
pivot_df_test = pivot_df_test.sort_index()
y_ed = np.concatenate([pivot_df['total_ed_visits'], pivot_df_test['total_ed_visits']])
all_dates = np.concatenate([pivot_df.index.values, pivot_df_test.index.values])

In [ ]:
all_dates.shape

In [ ]:
ed_visits_pp =  trace_ed.posterior_predictive[f"ed_visits"]
ed_visits_mean = np.median(ed_visits_pp.values,axis=(0,1))#.mean(axis=(0,1))  
ed_visits_hdi = az.hdi(ed_visits_pp.values, hdi_prob=0.95)  

forecast_ed = pp_forecast.predictions[f"forecast_ed_visits"]  
forecast_ed_mean = np.median(forecast_ed.values,axis=(0,1))#.mean(axis=(0,1))   
forecast_ed_hdi = az.hdi(forecast_ed.values, hdi_prob=0.95)  

forecast_latent_ed = pp_forecast.predictions[f"latent_forecast_ed"]
forecast_latent_ed_mean = np.median(forecast_latent_ed.values,axis=(0,1)).T#.mean(axis=(0,1))  
forecast_latent_ed_hdi = az.hdi(forecast_latent_ed.values, hdi_prob=0.95).transpose(1,0,2)  

latent_ed = trace_ed.posterior[f"latent_ed"]
latent_ed_mean = np.median(latent_ed.values,axis=(0,1)).T#.mean(axis=(0,1))    # shape: (T, regions)
latent_ed_hdi = az.hdi(latent_ed.values, hdi_prob=0.95).transpose(1,0,2)  # shape: (T(4,2), regions, 2)

ed_full_mean = np.concatenate([ed_visits_mean, forecast_ed_mean], axis=0)  # (T+H, R)
ed_full_hdi = np.concatenate([ed_visits_hdi, forecast_ed_hdi], axis=0)

latent_ed_full_mean = np.concatenate([latent_ed_mean, forecast_latent_ed_mean], axis=0)  # (T+H, R)
latent_ed_full_hdi = np.concatenate([latent_ed_hdi, forecast_latent_ed_hdi], axis=0)

for r in range(num_regions):
    fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    ax1=ax[0]

    ax1.plot(all_dates[1:], y_ed[1:, r], label='Observed ED', color='black')
    ax1.plot(all_dates[1:], ed_full_mean[1:, r], label='Predicted Mean', color='blue')
    ax1.fill_between(
        all_dates[1:],
        ed_full_hdi[1:, r, 0],
        ed_full_hdi[1:, r, 1],
        color='blue',
        alpha=0.3,
        label='95% Credible Interval'
    )

    ax2=ax[1]

    ax2.plot(all_dates[1:], latent_ed_full_mean[1:, r], label='Latent', color='red')
    ax2.fill_between(
        all_dates[1:],
        latent_ed_full_hdi[1:, r, 0],
        latent_ed_full_hdi[1:, r, 1],
        color='red',
        alpha=0.3,
        label='95% Credible Interval'
    )
    ax1.axvline(x=pivot_df_test.index[0], color='red', linestyle='--', label='Forecasting')
    ax2.axvline(x=pivot_df_test.index[0], color='red', linestyle='--', label='Forecasting')
    ax1.set_ylabel("ED Visits", color='black', fontsize=18)
    #ax2.set_ylabel("Predicted Cases", color='blue')
    ax1.set_title(f"Region {wwtps[r]}: ED Visits", fontsize = 24)
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.set_xlabel("Date", fontsize = 22)

In [ ]:
suffix="covid"
ed_visits_pp =  trace_ed.posterior[f"ed_contribution_{suffix}"]
ed_visits_mean = ed_visits_pp.values.mean(axis=(0,1))  
ed_visits_hdi = az.hdi(ed_visits_pp.values, hdi_prob=0.95)  

forecast_ed = pp_forecast.predictions[f"ed_contribution_{suffix}"]
forecast_ed_mean = forecast_ed.values.mean(axis=(0,1)) 
forecast_ed_hdi = az.hdi(forecast_ed.values, hdi_prob=0.95)  

forecast_latent_ed = pp_forecast.predictions[f"ed_contribution_residual"]
forecast_latent_ed_mean = forecast_latent_ed.values.mean(axis=(0,1))
forecast_latent_ed_hdi = az.hdi(forecast_latent_ed.values, hdi_prob=0.95)

latent_ed = trace_ed.posterior[f"ed_contribution_residual"]
latent_ed_mean = latent_ed.values.mean(axis=(0,1))  # shape: (T, regions)
latent_ed_hdi = az.hdi(latent_ed.values, hdi_prob=0.95)  # shape: (T(4,2), regions, 2)

ed_full_mean = np.concatenate([ed_visits_mean, forecast_ed_mean], axis=0)  # (T+H, R)
ed_full_hdi = np.concatenate([ed_visits_hdi, forecast_ed_hdi], axis=0)

latent_ed_full_mean = np.concatenate([latent_ed_mean, forecast_latent_ed_mean], axis=0)  # (T+H, R)
latent_ed_full_hdi = np.concatenate([latent_ed_hdi, forecast_latent_ed_hdi], axis=0)

for r in range(num_regions):
    fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    ax1=ax[0]
    ax1.plot(all_dates[1:], ed_full_mean[1:, r], label='Ed Contribution covid', color='blue')
    ax1.fill_between(
        all_dates[1:],
        ed_full_hdi[1:, r, 0],
        ed_full_hdi[1:, r, 1],
        color='blue',
        alpha=0.3,
        label='95% Credible Interval'
    )

    ax2=ax[1]

    ax2.plot(all_dates[1:], latent_ed_full_mean[1:, r], label='ED Contributions Residual', color='red')
    ax2.fill_between(
        all_dates[1:],
        latent_ed_full_hdi[1:, r, 0],
        latent_ed_full_hdi[1:, r, 1],
        color='red',
        alpha=0.3,
        label='95% Credible Interval'
    )
    ax1.axvline(x=pivot_df_test.index[0], color='red', linestyle='--', label='Forecasting')
    ax2.axvline(x=pivot_df_test.index[0], color='red', linestyle='--', label='Forecasting')
    ax1.set_ylabel(f"ED Contributions {suffix}", color='black', fontsize=18)
    ax2.set_ylabel("Ed Contributions Residual", color='blue')
    ax1.set_title(f"Region {wwtps[r]}: ED Visits", fontsize = 24)
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.set_xlabel("Date", fontsize = 22)

In [ ]:
diseases = ["covid", "rsv", "flua"]
ed_contrib = {
    d: trace_ed.posterior[f"ed_contribution_{d}"].values.mean(axis=(0, 1))  # shape (T, R)
    for d in diseases + ['residual']
}
# Remove last 21 days from the range
max_date_exclusive = max_date - pd.Timedelta(days=1)
full_range = pd.date_range(start=min_date, end=max_date_exclusive, freq="D")
filtered_range = full_range[:-21]  # drops the last 21 days
dfs = []
for key, arr in ed_contrib.items():
  temp = pd.DataFrame(arr, columns = wwtps)
  temp.insert(0, "time", filtered_range)  # or range(742)
  temp.insert(0,"target",key)
  dfs.append(temp)
final_df = pd.concat(dfs, ignore_index = True)
melt_df = pd.melt(final_df, id_vars=['time','target'], value_vars=wwtps)
melt_df.rename(columns={"variable":"wwtp", "value":"ed_contribution"}, inplace=True)
melt_df["ed_contribution_total"] = melt_df.groupby(["time", "wwtp"])["ed_contribution"].transform("sum")

# monthly aggregation
melt_df["month"] = melt_df["time"].values.astype("datetime64[M]")
monthly_df = melt_df.groupby(["target","wwtp","month"])[["ed_contribution",
"ed_contribution_total"]].sum().reset_index()
monthly_df['month'] = pd.to_datetime(monthly_df['month'])

# Sort to ensure correct order
monthly_df = monthly_df.sort_values(['wwtp', 'month', 'target'])

# Pivot to get targets as columns, summing ed_contribution
pivot_df = monthly_df.pivot_table(
    index=['wwtp', 'month'],
    columns='target',
    values='ed_contribution',
    aggfunc='sum',
    fill_value=0
).reset_index()

pivot_df['total'] = pivot_df[["covid","flua","residual","rsv"]].sum(axis=1)
pivot_df["percent_covid"] = 100*(pivot_df["covid"]/pivot_df["total"])
pivot_df["percent_rsv"] = 100*(pivot_df["rsv"]/pivot_df["total"])
pivot_df["percent_flua"] = 100*(pivot_df["flua"]/pivot_df["total"])
pivot_df["percent_residual"] = 100*(pivot_df["residual"]/pivot_df["total"])

import warnings
warnings.filterwarnings("ignore", message="This figure includes Axes that are not compatible with tight_layout")

wwtp_list = pivot_df['wwtp'].unique()

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8.6, 4 * 2), sharey=True,
sharex=True, gridspec_kw={"hspace": 0.2, "wspace": 0.2})
axes = axes.flatten()
pivot_df = pivot_df[["wwtp","month","percent_covid","percent_flua","percent_rsv","percent_residual"]]
light_colors = ["#A1D6E2", "#FFB7B2", "#C3B1E1", "#FFE6A7"]  # e.g. covid, rsv, flua, residual
for i,wwtp in enumerate(wwtp_list):
    ax = axes[i]
    ax2 = ax.twinx()
    data = pivot_df[pivot_df['wwtp'] == wwtp]
    data = data.set_index('month')
    # daily line plot
    df_daily = (df_train[df_train["wwtp"] == wwtp]
                .set_index('surveillance_date')
                .sort_index())

    # choose the order and ensure columns exist
    targets = ["percent_covid", "percent_flua", "percent_rsv", "percent_residual"]
    targets = [c for c in targets if c in data.columns]

    bottom = np.zeros(len(data))
    # width in days (since x is datetime); ~25 days looks like a “monthly bar”
    bar_width = 25

    for j, col in enumerate(targets):
        ax.bar(data.index, data[col].values, bottom=bottom,
               width=bar_width, color=light_colors[j], label=col, align='center')
        bottom += data[col].values

    y = df_daily['total_ed_visits'].rolling(7, min_periods=1).mean()
    ax2.plot(y.index, y.values, linewidth=1.5, color='black', label='Daily ED (7d MA)')
    ax.set_title(wwtp, fontsize=14)   # <- title per subplot
    # Date ticks every 2 months
    xt = data.index[::2]
    ax.set_xticks(xt)
    ax.set_xticklabels([d.strftime('%Y-%m') for d in xt], rotation=45, ha='right')
    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha('right')

handles1, labels1 = axes[0].get_legend_handles_labels()
line_handle = axes[0].plot([], [], color='black', label='Daily ED (7d MA)')[0]
fig.legend(handles1 + [line_handle],
           labels1 + ['Daily ED (7d MA)'],
           loc='upper center', bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False)

fig.supxlabel("Month", fontsize=12)
fig.text(0.05, 0.5, "(%) ED Contribution", va='center', rotation='vertical', fontsize=12)
fig.supylabel("Daily ED (7d MA)", fontsize=12, x=0.96)      # right side

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from scores.probability import crps_for_ensemble
import xarray as xr

y_obs = pivot_df_test['total_ed_visits']
y_obs.index.name = "time"

obs_array = xr.DataArray(
    data= y_obs.values,  # shape (T, R)
    dims=["time", "wwtp"],
    coords={
        "time": y_obs.index,         # surveillance_date
        "wwtp": y_obs.columns      # Annacis, Iona, etc.
    }
)

forecast_samples = (
    pp_forecast.predictions["forecast_ed_visits"]
    .stack(sample=("chain", "draw"))
    .transpose("sample", "forecast_ed_visits_dim_0", "forecast_ed_visits_dim_1")
    .rename({
        "forecast_ed_visits_dim_0": "time",
        "forecast_ed_visits_dim_1": "wwtp"
    })
)

forecast_samples["time"] = obs_array["time"]
forecast_samples = forecast_samples.assign_coords(
    wwtp=obs_array.coords["wwtp"].values
)


crps_matrix = crps_for_ensemble(
    obs=obs_array,                    # shape (time, wwtp)
    fcst=forecast_samples,            # shape (sample, time, wwtp)
    ensemble_member_dim="sample",     # must match forecast_samples dim
    preserve_dims=["time","wwtp"]   # must match both obs and forecast_samples
)

crps_mean = crps_matrix.mean(dim="time")
crps_df = crps_mean.to_dataframe(name="CRPS Bayes")

crps_df

In [ ]:
savepath = root_dir / "reports" / "tables"
crps_df.reset_index().to_csv(os.path.join(savepath,"crps_tarnished.csv"), index=False)

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import convolve

# Step 1–6: Full pipeline to compute and summarize disease-specific ED contributions from posterior
def compute_disease_contributions(trace, diseases, kernel_params, y_ed, forecast_dates, population, wwtps):    
    def combine_chains(samples):
        C, D, *rest = samples.shape
        return samples.reshape(C * D, *rest)

    def gamma_kernel(lag, shape, scale, delay):
        i = np.arange(lag)
        shifted_i = i - delay
        mask = shifted_i >= 0
        valid_kernel = np.zeros_like(shifted_i, dtype=float)
        valid_kernel[mask] = shifted_i[mask]**(shape - 1) * np.exp(-shifted_i[mask] / scale)
        kernel = valid_kernel / valid_kernel.sum()
        return kernel

    T, R = y_ed.shape
    S = trace.posterior.dims["chain"] * trace.posterior.dims["draw"]

    latent_ed = combine_chains(trace.posterior["latent_ed"].values)  # (S, T, R)
    latent_ed = latent_ed.transpose(0, 2, 1)  # Now (S, T, R)
    ed_noise = np.exp(latent_ed)*population[None, None, :]
    ed_rate_disease = {}
    for d in diseases:
        latent_d = combine_chains(trace.posterior[f"latent_{d}"].values)  # (S, T, R)
        nu_d = combine_chains(trace.posterior[f"nu_{d}"].values)          # (S, R)
        shape, scale = kernel_params[d]

        # Step 1: Get posterior delays
        delay_samples = combine_chains(trace.posterior[f"offset_ed_{d}"].values)  # shape (S,)

        # Step 2: Clip and round delays to integers
        delay_values = np.clip(delay_samples, 0, T - 1)

        # Step 3: Convolve using delay-specific kernels per sample
        conv = np.zeros((S, T, R))
        for s in range(S):
            kernel = gamma_kernel(T, shape, scale, delay_values[s])
            for r in range(R):
                conv[s, :, r] = convolve(latent_d[s, :, r], kernel, mode='same')

        ed_rate_disease[d] = np.exp(conv + nu_d[:, None, :])* population[None, None, :]  # (S, T, R)

    # Compute total rate across diseases + noise
    ed_rate_disease["noise"] = ed_noise
    total_rate = sum(ed_rate_disease.values()) # (S, T, R)
    all_cont = diseases + ["noise"]
    fraction_disease = {d: ed_rate_disease[d] / (total_rate + 1e-9) for d in all_cont}
    # Compute posterior disease contributions
    contrib_summary = []
    for d in all_cont:
        contrib_samples = fraction_disease[d] * y_ed.values[None, :, :]  # (S, T, R)
        contrib_mean = contrib_samples.mean(axis=0)
        contrib_hdi = np.percentile(contrib_samples, [2.5, 97.5], axis=0)

        for t in range(T):
            for r in range(R):
                contrib_summary.append({
                    "date": forecast_dates[t],
                    "region": wwtps[r],
                    "disease": d,
                    "mean": contrib_mean[t, r],
                    "ci_lower": contrib_hdi[0, t, r],
                    "ci_upper": contrib_hdi[1, t, r],
                    "obs": y_ed.values[t, r]
                })

    df_contrib = pd.DataFrame(contrib_summary)
    df_contrib["month"] = pd.to_datetime(df_contrib["date"]).dt.to_period("M")

    monthly_contrib = (
        df_contrib.groupby(["month", "disease"])
        .agg(mean_contrib=("mean", "sum"),
             ci_lower=("ci_lower", "sum"),
             ci_upper=("ci_upper", "sum"),
             obs_total=("obs", "sum")
            )
        .reset_index()
    )
    
    return df_contrib, monthly_contrib


In [ ]:
df_contrib, monthly_contrib = compute_disease_contributions(
    trace=trace_ed,                      # your ArviZ trace
    diseases=["covid", "rsv", "flua"],
    kernel_params={
        "covid": (3.0, 1.5),
        "rsv": (3.0, 1.5),
        "flua": (2.0, 1.8)
    },
    y_ed=pivot_df['total_ed_visits'],           # shape (T, R)
    forecast_dates=pivot_df.index.values,   # array of datetime, shape (T,)
    population=pop.population.values,         # array of population, shape (R,)
    wwtps=wwtps                       # list of region names, length R
)


In [ ]:
monthly_contrib

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.barplot(data=monthly_contrib, x="month", y="mean_contrib", hue="disease")
plt.xticks(rotation=45)
plt.ylabel("Monthly ED Contribution")
plt.title("Relative Disease Contribution to ED Visits per Month")
plt.tight_layout()
plt.show()


In [ ]:
# Filter out noise
df_disease_only = monthly_contrib[monthly_contrib["disease"] != "noise"].copy()

# Compute total contribution per month (diseases only)
total_by_month = (
    df_disease_only.groupby("month")["mean_contrib"]
    .transform("sum")
)

# Add a new column with the percentage
df_disease_only["percent_disease_only"] = 100 * df_disease_only["mean_contrib"] / total_by_month


In [ ]:
df_disease_only

In [ ]:
import matplotlib.pyplot as plt

# Pivot the DataFrame to get diseases as columns
df_percent_pivot = df_disease_only.pivot(index="month", columns="disease", values="percent_disease_only")

# Sort months (optional, but useful for consistency)
df_percent_pivot = df_percent_pivot.sort_index()

# Plot stacked bar chart
ax = df_percent_pivot.plot(kind="bar", stacked=True, figsize=(14, 6), colormap="Set2")

# Customize plot
ax.set_ylabel("Percent Contribution (%)", fontsize=18)
ax.set_xlabel("Month", fontsize=16)
ax.set_title("Relative Monthly % Contribution to ED Visits by Disease", fontsize=20)
ax.legend(title="Disease", fontsize=14, title_fontsize=16,  loc="upper right")
ax.set_ylim(0, 100)
plt.xticks(rotation=45, ha='right', fontsize = 16)
plt.xticks(fontsize = 16)
plt.tight_layout()
plt.show()


In [ ]:
az.plot_pair(trace_ed, var_names = ["offset_ed_covid", "offset_ed_flua", "offset_ed_rsv"], kind="kde");

In [ ]:
az.summary(trace_ed, var_names=["offset_ed_covid", "offset_ed_rsv", "offset_ed_flua"])